In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from codecarbon import EmissionsTracker
from xgboost import XGBClassifier

In [2]:
# --- Load and prep ---
df = pd.read_csv("HRDataset_cleaned.csv")
df['Age'] = df['Age'].abs()
leakage_cols = [col for col in df.columns if col.startswith('EmploymentStatus_')]
df = df.drop(columns=leakage_cols + ['State', 'EmpStatusID', 'Tenure_Years', 'Years_Since_Last_Review'])

X = df.drop(columns=['Termd'])
y = df['Termd']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
# --- Get top 10 features ---
rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_temp.fit(X_train, y_train)
top10_cols = pd.Series(rf_temp.feature_importances_, index=X_train.columns).nlargest(10).index.tolist()

In [4]:
df.columns

Index(['MarriedID', 'FromDiversityJobFairID', 'Salary', 'Termd', 'Sex',
       'HispanicLatino', 'EngagementSurvey', 'EmpSatisfaction',
       'SpecialProjectsCount', 'DaysLateLast30', 'Absences',
       'Internal_Transfer_Request', 'Avg_Overtime_Hours',
       'Distance_From_Home_Km', 'Remote_Work_Frequency', 'Age',
       'MaritalDesc_Married', 'MaritalDesc_Separated', 'MaritalDesc_Single',
       'MaritalDesc_Widowed', 'CitizenDesc_Non-Citizen',
       'CitizenDesc_US Citizen', 'RaceDesc_Asian',
       'RaceDesc_Black or African American', 'RaceDesc_Hispanic',
       'RaceDesc_Two or more races', 'RaceDesc_White',
       'Department_Executive Office', 'Department_IT/IS',
       'Department_Production       ', 'Department_Sales',
       'Department_Software Engineering', 'Position_Administrative Assistant',
       'Position_Area Sales Manager', 'Position_BI Developer',
       'Position_BI Director', 'Position_CIO', 'Position_Data Analyst',
       'Position_Data Analyst ', 'Position_D

In [5]:
## --- Other Top 10 features ---
top10_colx_2 = [
    "RecruitmentSource_Google Search",
    "Department_Production       ",
    "FromDiversityJobFairID",
    "RecruitmentSource_Diversity Job Fair",
    "SpecialProjectsCount",
    "Remote_Work_Frequency",
    "DaysLateLast30",
    "MaritalDesc_Single",
    "Department_IT/IS",
    "RecruitmentSource_Indeed"
]

top10_colx_2 =  np.array(top10_colx_2)

In [6]:
# --- Compare 3 models: Logistic (frugal), RF top10 (frugal), RF full (heavy) ---
models = {
    "Logistic Regression (10 features)": {
        "model": LogisticRegression(max_iter=2000, random_state=42),
        "X_train": X_train[top10_cols],
        "X_test": X_test[top10_cols],
        "scale": True
    },
    "Random Forest (10 features)": {
        "model": RandomForestClassifier(n_estimators=100, random_state=42),
        "X_train": X_train[top10_cols],
        "X_test": X_test[top10_cols],
        "scale": False
    },
    "Random Forest (73 features)": {
        "model": RandomForestClassifier(n_estimators=100, random_state=42),
        "X_train": X_train,
        "X_test": X_test,
        "scale": False
    },
    "Logistic Regression (Balanced)": {
        "model": LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced', random_state=42),
        "X_train": X_train[top10_cols],
        "X_test": X_test[top10_cols],
        "scale": True
    },
    "Random Forest (Tuned Frugal)": {
        "model": RandomForestClassifier(
            n_estimators=200, 
            max_depth=8, 
            min_samples_leaf=5,
            class_weight='balanced_subsample', # Spécifique au RF
            random_state=42
        ),
        "X_train": X_train[top10_cols],
        "X_test": X_test[top10_cols],
        "scale": False
    },
    "Logistic Regression (Number 2)": {
        "model": LogisticRegression(max_iter=2000, random_state=42),
        "X_train": X_train[top10_colx_2],
        "X_test": X_test[top10_colx_2],
        "scale": True
    },
    "Random Forest (Number 2)": {
        "model": RandomForestClassifier(n_estimators=100, random_state=42),
        "X_train": X_train[top10_colx_2],
        "X_test": X_test[top10_colx_2],
        "scale": False
    },
    "XGBoost (Number 2)": {
        "model": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42),
        "X_train": X_train[top10_colx_2],
        "X_test": X_test[top10_colx_2],
        "scale": False
    },
    "Logistic Regression (Balanced Number 2)": {
        "model": LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced', random_state=42),
        "X_train": X_train[top10_colx_2],
        "X_test": X_test[top10_colx_2],
        "scale": True
    },
    "Random Forest (Tuned Frugal Number 2)": {
        "model": RandomForestClassifier(
            n_estimators=200, 
            max_depth=8, 
            min_samples_leaf=5,
            class_weight='balanced_subsample', # Spécifique au RF
            random_state=42
        ),
        "X_train": X_train[top10_colx_2],
        "X_test": X_test[top10_colx_2],
        "scale": False
    }
}

In [7]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, roc_auc_score
import time
import pandas as pd

print("=== Frugal AI Comparison: Performance & Ethics ===\n")
# Mise à jour de l'en-tête pour inclure les nouvelles métriques
header = f"{'Model':<30} {'Feat.':>6} {'AUC':>6} {'Acc':>6} {'F1':>6} {'Prec':>6} {'Time(s)':>9} {'CO2(kg)':>10}"
print(header)
print("-" * len(header))

for name, config in models.items():
    Xtr = config["X_train"]
    Xte = config["X_test"]
    
    # Scaling si nécessaire
    if config["scale"]:
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        Xtr = pd.DataFrame(scaler.fit_transform(Xtr), columns=Xtr.columns)
        Xte = pd.DataFrame(scaler.transform(Xte), columns=Xte.columns)
    
    # Tracking des émissions et du temps
    tracker = EmissionsTracker(project_name=name, log_level="error")
    tracker.start()
    t0 = time.time()
    
    config["model"].fit(Xtr, y_train)
    
    train_time = time.time() - t0
    emissions = tracker.stop()
    
    # Calcul des prédictions
    proba = config["model"].predict_proba(Xte)[:, 1]
    preds = config["model"].predict(Xte) # Prédictions binaires (0 ou 1)
    
    # Calcul des métriques
    auc = roc_auc_score(y_test, proba)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    n_features = Xtr.shape[1]
    
    # Affichage formaté
    print(f"{name:<30} {n_features:>6} {auc:>6.2f} {acc:>6.2f} {f1:>6.2f} {prec:>6.2f} {train_time:>9.3f} {emissions:>10.6f}")


[codecarbon WARNING @ 09:17:09] Multiple instances of codecarbon are allowed to run at the same time.


=== Frugal AI Comparison: Performance & Ethics ===

Model                           Feat.    AUC    Acc     F1   Prec   Time(s)    CO2(kg)
--------------------------------------------------------------------------------------
Logistic Regression (10 features)     10   0.71   0.75   0.47   0.78     0.444   0.000000
Random Forest (10 features)        10   0.69   0.68   0.33   0.56     0.351   0.000000
Random Forest (73 features)        73   0.77   0.67   0.22   0.50     0.264   0.000000
Logistic Regression (Balanced)     10   0.71   0.70   0.56   0.55     0.009   0.000000
Random Forest (Tuned Frugal)       10   0.67   0.71   0.44   0.64     0.582   0.000000
Logistic Regression (Number 2)     10   0.78   0.73   0.48   0.67     0.025   0.000000
Random Forest (Number 2)           10   0.80   0.75   0.60   0.63     0.317   0.000000
XGBoost (Number 2)                 10   0.80   0.71   0.44   0.64     0.273   0.000000
Logistic Regression (Balanced Number 2)     10   0.79   0.71   0.62   0.56 

In [ ]:
### Le meilleur modèle est Random Forest (Tuned Frugal Number 2) 